# T03 — 운동역학과 역동역학: 지면반력, 관절 모멘트, joint power와 그 한계

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait-theory/note-T03.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait-theory/note-T03.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 본문

### 2. link-segment 모델과 자유물체도

아래 코드는 이 노트를 위해 만든 정적 예입니다. 70 kg인 사람이 한 발로 가만히 서 있고, COP가 발목보다 5 cm 앞에 있습니다.

In [ ]:
import numpy as np
g = 9.81
# 가상 예: 체질량 70 kg인 사람이 오른쪽을 보고 한 발로 가만히 서 있다 (X 앞+, Y 위+, 반시계 +)
M_body, m_foot = 70.0, 1.0
ankle = np.array([0.00, 0.08])      # 발목 관절 중심 (m)
com_f = np.array([0.03, 0.05])      # 발 질량중심
cop   = np.array([0.05, 0.00])      # 압력중심
F_grf = np.array([0.0, M_body * g]) # 정적이므로 지면반력 = 체중
a, alpha, I_f = np.zeros(2), 0.0, 0.004   # 가속도 0 (정적)

def cross2(r, F):                    # 2D 외적의 z 성분
    return r[0]*F[1] - r[1]*F[0]

# 발의 자유물체도: 원위단 = 지면반력(COP), 근위단 = 발목의 반력 R과 모멘트 M (미지수)
R = m_foot*a - F_grf + np.array([0.0, m_foot*g])          # ΣF = m a
M = I_f*alpha - cross2(cop - com_f, F_grf) - cross2(ankle - com_f, R)   # ΣM(COM 기준) = I α
print("발목 반력 R (N):", np.round(R, 1))
print("발목 내부 모멘트 M (N·m):", round(M, 1), "← 음수 = 시계 방향 = 족저굴곡근 모멘트")
print("단순 벡터법(지면반력 × 발목까지 수평거리):", round(F_grf[1]*(cop[0]-ankle[0]), 1), "N·m (외부 배측굴곡 모멘트)")

# 관절 파워 P = M·ω 의 부호 (W)
for name, Mj, w in [("push-off 발목", -80.0, -3.0), ("LR 무릎", 30.0, -2.0), ("TSt 발목", -60.0, 1.0)]:
    print(f"{name:10s} M={Mj:+6.1f} N·m, ω={w:+4.1f} rad/s → P={Mj*w:+7.1f} W")